# 🌲 ML Project — Random Forest para Pitch Shifting (Octave Up)

## Objetivo

Entrenar un modelo **Random Forest Regressor** capaz de transformar señales de guitarra eléctrica aplicando **Pitch Shift +1 octava**, operando directamente sobre la forma de onda.

Este notebook es el **modelo de comparación** frente al WaveNet. A diferencia del WaveNet:

- Usa **Random Forest** (ensamble de árboles de decisión) — ML clásico, sin redes neuronales
- Trabaja también en el **dominio del tiempo** (raw waveform) para comparación justa
- Usa `MultiOutputRegressor` de scikit-learn para manejar salidas de 512 dimensiones
- **Interpretable**: permite ver importancia de features por posición del buffer

## Flujo

1. Instalación de dependencias
2. Configuración global (idéntica al notebook WaveNet)
3. Preparación del dataset
4. Entrenamiento del Random Forest
5. Inferencia y reconstrucción de audio
6. Comparación de métricas vs WaveNet

## ⚙️ 1. Instalación de dependencias

In [1]:
!pip install -q librosa soundfile scikit-learn joblib

import subprocess, sys
pkgs = ["librosa", "scikit-learn", "numpy", "scipy", "joblib"]
for pkg in pkgs:
    result = subprocess.run([sys.executable, "-m", "pip", "show", pkg],
                            capture_output=True, text=True)
    for line in result.stdout.splitlines():
        if line.startswith("Version"):
            print(f"{pkg:15}: {line}")

librosa        : Version: 0.11.0
scikit-learn   : Version: 1.6.1
numpy          : Version: 2.4.6
scipy          : Version: 1.16.3
joblib         : Version: 1.5.3


## 📦 2. Imports

In [2]:
import gc
import os
import time
import numpy as np
import librosa
import soundfile as sf
import matplotlib.pyplot as plt
import IPython.display as ipd

from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error
import joblib

print("scikit-learn :", __import__('sklearn').__version__)
print("librosa      :", librosa.__version__)
print("numpy        :", np.__version__)

scikit-learn : 1.6.1
librosa      : 0.11.0
numpy        : 2.4.6


## ⚙️ 3. Configuración global

**Idéntica al notebook WaveNet** para garantizar comparación justa.

In [3]:
# ── Audio ────────────────────────────────────────────────────────────────
SAMPLE_RATE    = 96000
BUFFER_SIZE    = 512          # muestras por buffer (~5.33 ms)
HOP_SIZE       = 256          # salto entrenamiento (50% overlap)
HOP_SIZE_INFER = 512          # salto inferencia (0% overlap)

# ── Rutas Kaggle ──────────────────────────────────────────────────────────
DATASET_PATH   = Path("/kaggle/input/datasets/jhorlandaniel/datasetparapitchshift/DataSetParaPitchShift")
X_PATH         = DATASET_PATH / "x"
Y_PATH         = DATASET_PATH / "y"
OUTPUT_PATH    = Path("/kaggle/working")
RF_MODEL_PATH  = str(OUTPUT_PATH / "best_wavenet_model.h5")

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# ── Hiperparámetros Random Forest ─────────────────────────────────────────
RF_PARAMS = {
    "n_estimators" : 100,    # número de árboles
    "max_depth"    : 20,     # profundidad máxima por árbol
    "min_samples_split": 5,  # mínimo de muestras para dividir nodo
    "min_samples_leaf" : 2,  # mínimo de muestras en hoja
    "max_features" : "sqrt", # features por split (sqrt(512) ≈ 22)
    "n_jobs"       : -1,     # usa todos los núcleos disponibles
    "random_state" : 42,
}

print(f"SAMPLE_RATE   : {SAMPLE_RATE} Hz")
print(f"BUFFER_SIZE   : {BUFFER_SIZE} samples  ({BUFFER_SIZE/SAMPLE_RATE*1000:.2f} ms)")
print(f"HOP_SIZE      : {HOP_SIZE}")
print(f"Dataset       : {DATASET_PATH}")
print(f"Carpeta x     : {X_PATH.exists()}")
print(f"Carpeta y     : {Y_PATH.exists()}")
print(f"Salida        : {OUTPUT_PATH}")

# ── Validación de rutas ───────────────────────────────────────────────────
if not X_PATH.exists() or not Y_PATH.exists():
    raise FileNotFoundError(
        f"\n❌ Dataset no encontrado en:\n   {DATASET_PATH}\n\n"
        "👉 En Kaggle, ve a la barra lateral → 'Add data' → busca:\n"
        "   'datasetparapitchshift' (by jhorlandaniel)\n"
        "   y agrégalo al notebook antes de ejecutar."
    )
print("✓ Dataset encontrado correctamente.")


SAMPLE_RATE   : 96000 Hz
BUFFER_SIZE   : 512 samples  (5.33 ms)
HOP_SIZE      : 256
Dataset       : /kaggle/input/datasets/jhorlandaniel/datasetparapitchshift/DataSetParaPitchShift
Carpeta x     : False
Carpeta y     : False
Salida        : /kaggle/working


## 📂 4. Emparejado automático x → y

Mismo proceso que en el notebook WaveNet para garantizar que usamos el mismo dataset.

In [4]:
x_files = sorted(X_PATH.glob("*.wav"))
pairs = []

for x_file in x_files:
    y_expected = Y_PATH / f"{x_file.stem}Shifted.wav"
    if y_expected.exists():
        pairs.append({"x": x_file, "y": y_expected})
    else:
        fallbacks = [f for f in Y_PATH.glob(f"{x_file.stem}*.wav")]
        if len(fallbacks) == 1:
            pairs.append({"x": x_file, "y": fallbacks[0]})
            print(f"  ⚠ Fallback: {x_file.name} → {fallbacks[0].name}")
        elif len(fallbacks) > 1:
            print(f"  ✗ Múltiples candidatos para {x_file.name} — omitido")
        else:
            print(f"  ✗ Sin objetivo para: {x_file.name}")

# Split a nivel de archivo (igual que WaveNet — sin data leakage)
if len(pairs) == 0:
    raise RuntimeError(
        "\n❌ No se encontraron pares x/y en el dataset.\n"
        "Verifica que las carpetas 'x' e 'y' contengan archivos .wav "
        "con los nombres esperados (ej: archivo.wav / archivoShifted.wav)."
    )

n_val = max(1, int(len(pairs) * 0.1))
# Garantizar al menos 1 par de entrenamiento
if n_val >= len(pairs):
    n_val = len(pairs) - 1

pairs_val = pairs[-n_val:]
pairs_tr  = pairs[:-n_val]

print(f"\nPares encontrados    : {len(pairs)}")
print(f"Pares entrenamiento  : {len(pairs_tr)}")
print(f"Pares validación     : {len(pairs_val)}")


Pares encontrados    : 0
Pares entrenamiento  : 0
Pares validación     : 0


## 🔧 5. Funciones utilitarias

Mismas funciones del notebook WaveNet — reutilizadas sin cambios.

In [5]:
def normalize_audio_pair(x_audio, y_audio):
    """
    Normaliza ambas señales por el máximo absoluto compartido.
    Preserva la relación de amplitud entre x e y.
    """
    shared_max = max(np.max(np.abs(x_audio)), np.max(np.abs(y_audio)))
    if shared_max < 1e-6:
        print("  ⚠ Par silencioso detectado — sin normalizar")
        return x_audio, y_audio
    return x_audio / shared_max, y_audio / shared_max


def create_buffers(signal, buffer_size, hop_size):
    """
    Segmenta una señal 1-D en chunks de tamaño buffer_size
    con salto hop_size entre chunks.
    Retorna: np.ndarray (N, buffer_size) float32
    """
    chunks = [
        signal[i : i + buffer_size]
        for i in range(0, len(signal) - buffer_size + 1, hop_size)
    ]
    return np.array(chunks, dtype=np.float32)


print("Funciones utilitarias listas.")

Funciones utilitarias listas.


## 📊 6. Carga y preparación del dataset

Para Random Forest necesitamos materializar los arrays en RAM — a diferencia de WaveNet que usaba `tf.data` con generadores.

> ⚠️ **Nota de memoria**: si el dataset es muy grande, se recomienda usar un subconjunto. El parámetro `MAX_BUFFERS_PER_FILE` limita cuántos buffers se toman de cada archivo.

In [6]:
MAX_BUFFERS_PER_FILE = 2000  # ajusta según la RAM disponible (None = todos)

def load_pairs_to_arrays(pairs_list, max_buffers=None):
    """
    Carga pares de audio y los convierte en arrays numpy.
    X: (N, BUFFER_SIZE) — waveform de entrada
    Y: (N, BUFFER_SIZE) — waveform objetivo (pitch shifted)
    """
    # ── Validación temprana ────────────────────────────────────────────────
    if not pairs_list:
        raise ValueError(
            "\n❌ pairs_list está vacío — no hay archivos de audio para cargar.\n"
            "Causas posibles:\n"
            "  1. El dataset no está montado (revisa Celda 3: 'Carpeta x: False')\n"
            "  2. Ningún archivo .wav de 'x' tiene su correspondiente en 'y'\n"
            "  3. El split dejó el conjunto de entrenamiento vacío"
        )
    X_all, Y_all = [], []

    for i, pair in enumerate(pairs_list):
        print(f"  Cargando {i+1}/{len(pairs_list)}: {pair['x'].name}", end="\r")

        x_audio, _ = librosa.load(str(pair["x"]), sr=SAMPLE_RATE, mono=True)
        y_audio, _ = librosa.load(str(pair["y"]), sr=SAMPLE_RATE, mono=True)

        x_audio, y_audio = normalize_audio_pair(x_audio, y_audio)

        min_len = min(len(x_audio), len(y_audio))
        x_bufs = create_buffers(x_audio[:min_len], BUFFER_SIZE, HOP_SIZE)
        y_bufs = create_buffers(y_audio[:min_len], BUFFER_SIZE, HOP_SIZE)

        n = min(len(x_bufs), len(y_bufs))
        if max_buffers is not None:
            n = min(n, max_buffers)

        X_all.append(x_bufs[:n])
        Y_all.append(y_bufs[:n])

        del x_audio, y_audio, x_bufs, y_bufs
        gc.collect()

    print("  ✓ Carga completada.                    ")

    # ── Validación post-carga ──────────────────────────────────────────────
    if len(X_all) == 0:
        raise ValueError(
            "\n❌ No se generó ningún buffer de audio.\n"
            "Verifica que los archivos .wav sean válidos y tengan "
            f"al menos {BUFFER_SIZE} muestras ({BUFFER_SIZE/SAMPLE_RATE*1000:.1f} ms)."
        )

    return np.vstack(X_all).astype(np.float32), np.vstack(Y_all).astype(np.float32)


print("Cargando datos de entrenamiento...")
t0 = time.time()
X_train, Y_train = load_pairs_to_arrays(pairs_tr, max_buffers=MAX_BUFFERS_PER_FILE)
print(f"X_train: {X_train.shape}  |  Y_train: {Y_train.shape}")
print(f"RAM aproximada: {X_train.nbytes / 1e6:.1f} MB (X) + {Y_train.nbytes / 1e6:.1f} MB (Y)")
print(f"Tiempo de carga: {time.time() - t0:.1f}s")

print("\nCargando datos de validación...")
X_val, Y_val = load_pairs_to_arrays(pairs_val, max_buffers=MAX_BUFFERS_PER_FILE)
print(f"X_val: {X_val.shape}  |  Y_val: {Y_val.shape}")

Cargando datos de entrenamiento...
  ✓ Carga completada.                    


ValueError: need at least one array to concatenate

## 🌲 7. Arquitectura del modelo Random Forest

### Estructura

```
Input: (N, 512)  — buffer de waveform
         │
   MultiOutputRegressor
         │
   RandomForestRegressor
   ├── Árbol 1  ──┐
   ├── Árbol 2  ──┤  promedio
   ├── ...      ──┤  de predicciones
   └── Árbol 100──┘
         │
Output: (N, 512) — waveform transformada
```

**`MultiOutputRegressor`** entrena un RF independiente **por cada una de las 512 posiciones de salida**. Esto permite al modelo aprender la relación de cada muestra de salida con las 512 muestras de entrada.

In [ ]:
rf_base = RandomForestRegressor(
    n_estimators      = RF_PARAMS["n_estimators"],
    max_depth         = RF_PARAMS["max_depth"],
    min_samples_split = RF_PARAMS["min_samples_split"],
    min_samples_leaf  = RF_PARAMS["min_samples_leaf"],
    max_features      = RF_PARAMS["max_features"],
    n_jobs            = RF_PARAMS["n_jobs"],
    random_state      = RF_PARAMS["random_state"],
    verbose           = 0,
)

model_rf = MultiOutputRegressor(rf_base, n_jobs=-1)

print("Modelo Random Forest listo.")
print(f"  Estimadores     : {RF_PARAMS['n_estimators']}")
print(f"  Profundidad máx : {RF_PARAMS['max_depth']}")
print(f"  Features/split  : {RF_PARAMS['max_features']}  (sqrt(512) ≈ 22)")
print(f"  Salidas         : {BUFFER_SIZE} (una por sample del buffer)")

## 🏋️ 8. Entrenamiento

In [ ]:
print("Iniciando entrenamiento...")
print(f"Muestras de entrenamiento : {X_train.shape[0]}")
print(f"Dimensión entrada         : {X_train.shape[1]}")
print(f"Dimensión salida          : {Y_train.shape[1]}")
print()

t_start = time.time()
model_rf.fit(X_train, Y_train)
t_train = time.time() - t_start

print(f"\n✓ Entrenamiento completado en {t_train:.1f}s  ({t_train/60:.1f} min)")

# Guardar modelo
joblib.dump(model_rf, RF_MODEL_PATH, compress=3)
print(f"✓ Modelo guardado en: {RF_MODEL_PATH}")

## 📈 9. Evaluación con métricas comparables al WaveNet

In [ ]:
def snr_db(y_true, y_pred):
    """Signal-to-Noise Ratio en dB. Mayor = mejor."""
    noise = y_true - y_pred
    signal_power = np.var(y_true, axis=1)
    noise_power  = np.var(noise,  axis=1)
    # Evitar división por cero en frames silenciosos
    mask = signal_power > 1e-10
    snr  = 10 * np.log10(signal_power[mask] / (noise_power[mask] + 1e-10))
    return float(np.mean(snr))


def pearson_corr(y_true, y_pred):
    """Correlación de Pearson promedio por frame. Rango [-1, 1], mayor = mejor."""
    corrs = [
        np.corrcoef(y_true[i], y_pred[i])[0, 1]
        for i in range(min(len(y_true), 500))  # muestra de 500 frames
    ]
    return float(np.nanmean(corrs))


print("Calculando métricas sobre validación...")
t0 = time.time()

Y_pred_val = model_rf.predict(X_val)
t_infer    = time.time() - t0

mse_val  = mean_squared_error(Y_val, Y_pred_val)
snr_val  = snr_db(Y_val, Y_pred_val)
corr_val = pearson_corr(Y_val, Y_pred_val)

print(f"\n{'='*45}")
print("MÉTRICAS — Random Forest (validación)")
print(f"{'='*45}")
print(f"  MSE (val_loss)  : {mse_val:.6f}")
print(f"  SNR             : {snr_val:.2f} dB")
print(f"  Correlación     : {corr_val:.4f}")
print(f"  Tiempo inferencia: {t_infer:.2f}s  ({len(X_val)} frames)")
print(f"{'='*45}")

## 🔍 10. Importancia de features

Una ventaja del Random Forest sobre WaveNet: podemos ver **qué posiciones del buffer de entrada son más relevantes** para predecir la salida.

In [ ]:
# Extraer importancias del primer estimador (el RF para la salida central)
mid_output = BUFFER_SIZE // 2
rf_central = model_rf.estimators_[mid_output]
importances = rf_central.feature_importances_

# Suavizado con ventana para visualización
window = 10
smooth = np.convolve(importances, np.ones(window)/window, mode='same')

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# ── Gráfico 1: importancias de features ───────────────────────────────────
axes[0].fill_between(range(BUFFER_SIZE), smooth, alpha=0.4, color='steelblue')
axes[0].plot(smooth, color='steelblue', linewidth=1.5)
axes[0].axvline(x=mid_output, color='red', linestyle='--', alpha=0.7,
                label=f'Sample central ({mid_output})')
axes[0].set_title(f'Importancia de features — predictor para output[{mid_output}]',
                  fontsize=13, fontweight='bold')
axes[0].set_xlabel('Posición en el buffer de entrada (sample)')
axes[0].set_ylabel('Importancia (Gini)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# ── Gráfico 2: comparación pred vs ground truth ───────────────────────────
idx = 0  # primer frame de validación
t   = np.arange(BUFFER_SIZE) / SAMPLE_RATE * 1000  # en milisegundos
axes[1].plot(t, Y_val[idx],      label='Ground Truth (y)',  color='seagreen',  alpha=0.8)
axes[1].plot(t, Y_pred_val[idx], label='RF Prediction',     color='coral',     alpha=0.8, linestyle='--')
axes[1].set_title('Waveform — Ground Truth vs Predicción RF (primer frame validación)',
                  fontsize=13, fontweight='bold')
axes[1].set_xlabel('Tiempo (ms)')
axes[1].set_ylabel('Amplitud')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(str(OUTPUT_PATH / "rf_analysis.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Gráficos guardados en rf_analysis.png")

## 🎧 11. Inferencia y reconstrucción de audio

Carga el modelo guardado y genera el audio con pitch shift +1 octava.
Usa **overlap-add** idéntico al WaveNet para comparación justa.

In [ ]:
# ── Cargar modelo guardado (opcional si ya está en memoria) ───────────────
# model_rf = joblib.load(RF_MODEL_PATH)

def predict_audio_rf(model, x_audio):
    """
    Transforma una señal de audio completa usando el modelo RF.
    Divide en buffers → predice → reconstruye con overlap-add.
    Mismo proceso que en el notebook WaveNet para comparación justa.
    """
    x_bufs = create_buffers(x_audio, BUFFER_SIZE, HOP_SIZE_INFER)
    preds  = model.predict(x_bufs)  # (N, BUFFER_SIZE)

    output = np.zeros(len(x_audio), dtype=np.float32)
    counts = np.zeros(len(x_audio), dtype=np.float32)

    for i, pred in enumerate(preds):
        start = i * HOP_SIZE_INFER
        end   = start + BUFFER_SIZE
        if end > len(x_audio):
            break
        output[start:end] += pred
        counts[start:end] += 1.0

    return (output / np.maximum(counts, 1.0)).astype(np.float32)


# ── Cargar audio de prueba ─────────────────────────────────────────────────
audio_path = DATASET_PATH / "x" / "Prueba - MAIN.wav"
if not audio_path.exists():
    audio_path = DATASET_PATH / "x" / "AcordesYArpegios.wav"

print(f"Audio de prueba: {audio_path.name}")
x_signal, _ = librosa.load(str(audio_path), sr=SAMPLE_RATE, mono=True)

print("🎸 Original:")
display(ipd.Audio(x_signal, rate=SAMPLE_RATE))

# ── Predicción ─────────────────────────────────────────────────────────────
print("Procesando con Random Forest...")
t0 = time.time()
y_rf = predict_audio_rf(model_rf, x_signal)
t_full = time.time() - t0
print(f"⏱ Tiempo de inferencia completo: {t_full:.2f}s")

print("🌲 Random Forest — Pitch Shift +1 Octava:")
display(ipd.Audio(y_rf, rate=SAMPLE_RATE))

# ── Guardar resultado ──────────────────────────────────────────────────────
rf_output_path = str(OUTPUT_PATH / "rf_output.wav")
sf.write(rf_output_path, y_rf, SAMPLE_RATE)
print(f"✓ Guardado: {rf_output_path}")

## 📊 12. Tabla comparativa final: Random Forest vs WaveNet

Completa la columna de WaveNet con los resultados del otro notebook.

In [ ]:
# ── Completar con los valores reales de tu notebook WaveNet ───────────────
# (copia el val_loss final, SNR y tiempo de inferencia del otro notebook)
WAVENET_RESULTS = {
    "MSE (val_loss)"    : None,   # <-- pegar aquí
    "SNR (dB)"          : None,   # <-- pegar aquí
    "Correlación"       : None,   # <-- pegar aquí
    "Tiempo inferencia" : None,   # <-- en segundos
}

RF_RESULTS = {
    "MSE (val_loss)"    : mse_val,
    "SNR (dB)"          : snr_val,
    "Correlación"       : corr_val,
    "Tiempo inferencia" : t_full,
}

print(f"{'Métrica':<22} {'Random Forest':>18} {'WaveNet':>18}")
print("-" * 60)
for metrica in RF_RESULTS:
    rf_val = RF_RESULTS[metrica]
    wn_val = WAVENET_RESULTS[metrica]
    rf_str = f"{rf_val:.6f}" if rf_val is not None else "—"
    wn_str = f"{wn_val:.6f}" if wn_val is not None else "(pendiente)"

    # Indicar qué modelo ganó en cada métrica
    if rf_val is not None and wn_val is not None:
        if metrica in ["SNR (dB)", "Correlación"]:
            winner = "🌲 RF" if rf_val > wn_val else "🧠 WaveNet"
        else:  # MSE y tiempo: menor es mejor
            winner = "🌲 RF" if rf_val < wn_val else "🧠 WaveNet"
        print(f"{metrica:<22} {rf_str:>18} {wn_str:>18}   ← {winner}")
    else:
        print(f"{metrica:<22} {rf_str:>18} {wn_str:>18}")

print("-" * 60)
print("\n🔍 Interpretación:")
print("  MSE / val_loss  → menor es mejor (igual métrica que WaveNet)")
print("  SNR (dB)        → mayor es mejor (calidad de la señal reconstruida)")
print("  Correlación     → mayor es mejor (similitud con el target)")
print("  Tiempo inferencia → menor es mejor (viabilidad en tiempo real)")